# Stage 3 – Partnership extraction

**Goal:** turn ball-by-ball data into one row per batting partnership.

The extractor (see `library/lib_02_partnership_extraction.ipynb`) reads every England innings ball by ball,
keeps track of the pair at the crease and closes a partnership when a wicket record, a recorded replacement or
the end of the innings says so.

**Reads:** `data/processed/match_metadata.csv` and the raw match files
**Writes:** `data/processed/partnership_observations.csv` (the audit trail of the whole project),
`innings_summary.csv`, `extraction_warnings.csv`

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_01_data_loading.ipynb
%run ./library/lib_02_partnership_extraction.ipynb
stage_log = get_stage_logger("03_extract")

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
extract_archive_if_needed()
included_matches = read_table(PROCESSED_FOLDER / "match_metadata.csv")
observations, innings_table, extraction_warnings = extract_all_matches(included_matches)

write_table(observations, PROCESSED_FOLDER / "partnership_observations.csv")
write_table(innings_table, PROCESSED_FOLDER / "innings_summary.csv")
write_table(extraction_warnings, PROCESSED_FOLDER / "extraction_warnings.csv")

### Summary of the extraction

In [3]:
stage_log.info("matches: %d, England innings: %d (qualifying %d), observations: %d, warnings: %d",
               len(included_matches), len(innings_table), int(innings_table["qualifying"].sum()),
               len(observations), len(extraction_warnings))
stage_log.info("observations by format:\n%s", observations["format"].value_counts().reindex(list(FORMATS)).to_string())
stage_log.info("why partnerships ended:\n%s", observations["boundary_reason"].value_counts().to_string())
for warning in extraction_warnings.to_dict("records"):
    stage_log.warning("extraction warning: %s", warning)

2026-11-16 20:27:19,827 INFO batnet.03_extract: matches: 287, England innings: 350 (qualifying 350), observations: 2828, warnings: 0


2026-11-16 20:27:19,828 INFO batnet.03_extract: observations by format:
format
Test    1357
ODI      708
T20I     763


2026-11-16 20:27:19,829 INFO batnet.03_extract: why partnerships ended:
boundary_reason
dismissal:caught               1642
dismissal:bowled                508
dismissal:lbw                   303
innings_end                     161
dismissal:run out                90
dismissal:caught and bowled      64
dismissal:stumped                55
dismissal:hit wicket              2
replacement                       2
retirement:retired hurt           1


In [4]:
# A look at the first partnerships of the table.
observations.head(10)

,match_id,source_file,match_date,format,innings_index,batting_team,partnership_seq,player_1_id,player_1_name,player_2_id,player_2_name,partnership_runs,batter_runs,extras_runs,deliveries,start_delivery_id,end_delivery_id,boundary_reason,quality_status
0,1185305,1185305.json,2020-01-03,Test,1,England,1,922e1b19,Z Crawley,97e434a6,DP Sibley,8,8,0,18,1.0.1,1.2.6,dismissal:caught,ok
1,1185305,1185305.json,2020-01-03,Test,1,England,2,97e434a6,DP Sibley,bdadf7da,JL Denly,55,51,4,131,1.3.1,1.24.5,dismissal:caught,ok
2,1185305,1185305.json,2020-01-03,Test,1,England,3,a343262c,JE Root,bdadf7da,JL Denly,42,41,1,96,1.24.6,1.40.5,dismissal:caught,ok
3,1185305,1185305.json,2020-01-03,Test,1,England,4,bdadf7da,JL Denly,e087956b,BA Stokes,22,22,0,40,1.40.6,1.47.3,dismissal:bowled,ok
4,1185305,1185305.json,2020-01-03,Test,1,England,5,e087956b,BA Stokes,e94bc520,OJ Pope,58,58,0,114,1.47.4,1.66.3,dismissal:caught,ok
5,1185305,1185305.json,2020-01-03,Test,1,England,6,99b75528,JC Buttler,e94bc520,OJ Pope,36,36,0,51,1.66.4,1.74.6,dismissal:caught,ok
6,1185305,1185305.json,2020-01-03,Test,1,England,7,e94915e6,SM Curran,e94bc520,OJ Pope,10,10,0,24,1.75.1,1.78.6,dismissal:bowled,ok
7,1185305,1185305.json,2020-01-03,Test,1,England,8,ad46c747,DM Bess,e94bc520,OJ Pope,0,0,0,7,1.79.1,1.80.1,dismissal:caught,ok
8,1185305,1185305.json,2020-01-03,Test,1,England,9,e94bc520,OJ Pope,eba6a852,SCJ Broad,3,3,0,11,1.80.2,1.81.6,dismissal:bowled,ok
9,1185305,1185305.json,2020-01-03,Test,1,England,10,d12143bf,JM Anderson,e94bc520,OJ Pope,35,33,2,61,1.82.1,1.91.5,dismissal:caught,ok
